[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shan4949/ml-visualized/blob/main/notebooks/09_naive_bayes_generative.ipynb)

# 09 · Naive Bayes & Generative Classifiers: Modeling How the Data Was Made

**Part 1 — Classical supervised learning** · Notebook 9 of 27

Logistic regression (notebook 08) models $p(y \mid x)$ directly: "given these features, how likely is each class?" There's a second, completely different way to build a classifier. Model **how each class generates its data**, $p(x \mid y)$, plus how common each class is, $p(y)$, and then flip it around with **Bayes' rule**. These are **generative** classifiers. They're often simpler to fit (frequently in closed form, by counting), they can generate new data, and they shed light on *why* logistic regression has the form it does: if the classes are Gaussian with a shared covariance, Bayes' rule produces **exactly** the logistic model.

**What you'll learn**

1. Discriminative vs. generative classifiers
2. The Bayes decision rule, and the Bayes error (the best any classifier can do)
3. Gaussian class-conditionals: fitting $\mu_k, \Sigma_k, \pi_k$ by MLE
4. **QDA**: curved (quadratic) decision boundaries
5. **LDA**: shared covariance gives straight boundaries, and the posterior is *exactly* logistic
6. **Naive Bayes**: the conditional-independence shortcut
7. A spam filter from scratch: Bernoulli and multinomial Naive Bayes, and Laplace smoothing
8. Naive Bayes is a linear classifier in disguise (CX 4240 HW3)
9. Generative vs. discriminative: who wins, and when (Ng & Jordan)
10. The generative superpower: sampling new data

*Course map: CX 4240 Lectures 8–10 (Naive Bayes, Gaussian class-conditionals, discriminative vs. generative classifiers) and HW3 (Bernoulli Naive Bayes is linear); ISYE 4600 W8 (Bayes classifier, LDA, QDA, Naive Bayes) and HW3 (Bayes classifier for spam filtering; decision boundary of a Bayesian classifier).*

In [1]:
#@title ⚙️ Setup — run this cell first (imports, plot style, helpers)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider

# One consistent, colorblind-checked palette used across the whole series
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100",
         magenta="#e87ba4", green="#008300", violet="#4a3aa7", red="#e34948",
         ink="#0b0b0b", muted="#52514e", light="#d9d8d3")
plt.rcParams.update({
    "figure.figsize": (6.4, 4.2), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.prop_cycle": plt.cycler(color=[C[k] for k in
        ["blue", "orange", "aqua", "yellow", "magenta", "green", "violet", "red"]]),
    "lines.linewidth": 2, "font.size": 11, "legend.frameon": False,
    "animation.embed_limit": 40, "animation.frame_format": "jpeg",   # JPEG frames keep notebooks small
})

def show_anim(anim):
    """Render a matplotlib animation as an interactive player (play / pause / scrub)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml(default_mode="once"))

rng = np.random.default_rng(0)
print("Setup complete ✓")
from scipy.stats import multivariate_normal, norm
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
print("Ready ✓")

CLS = [C["blue"], C["red"], C["aqua"]]
def region_plot(ax, predict_proba, X, y, xlim, ylim, n=220, K=2):
    g1, g2 = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    P = predict_proba(np.column_stack([g1.ravel(), g2.ravel()]))
    if K == 2:
        ax.contourf(g1, g2, P[:, 1].reshape(g1.shape), levels=np.linspace(0, 1, 21), cmap="RdBu_r", alpha=0.6)
        ax.contour(g1, g2, P[:, 1].reshape(g1.shape), levels=[0.5], colors=C["ink"], linewidths=2)
    else:
        ax.contourf(g1, g2, P.argmax(1).reshape(g1.shape), levels=np.arange(K + 1) - 0.5, colors=CLS[:K], alpha=0.3)
    for k in range(K):
        ax.scatter(*X[y == k].T, s=14, color=CLS[k], edgecolor="white", linewidth=0.5)
    ax.set(xlim=xlim, ylim=ylim); ax.grid(False)

Setup complete ✓


Ready ✓


---
## 1. Discriminative vs. generative

CX 4240 Lecture 10 frames the split like this. Both kinds of classifier end up computing the posterior $p(y \mid x)$ and predicting the most probable class. They differ in how they get there:

| | **Discriminative** (logistic regression, SVM, neural nets) | **Generative** (Naive Bayes, LDA, QDA) |
|---|---|---|
| Models | $p(y \mid x)$ directly | $p(x \mid y)$ and $p(y)$ |
| Then | done | Bayes' rule: $p(y\mid x) = \dfrac{p(x\mid y)\,p(y)}{\sum_{y'} p(x\mid y')\,p(y')}$ |
| Question it answers | "where's the boundary?" | "what does each class look like?" |
| Fitting | usually iterative (gradient descent) | often closed form (averages, counts) |
| Can generate new $x$? | no | yes |
| Handles missing features? | awkwardly | naturally (marginalize them out) |

An analogy: to tell cats from dogs, a discriminative model learns "pointy ears and whiskers → cat." A generative model learns "here's what cats look like; here's what dogs look like," and classifies a new animal by asking which description fits it better.

---
## 2. The Bayes decision rule and the Bayes error

Suppose for a moment we **knew** the true class-conditional distributions and priors. ISYE 4600 (W8) gives the rule: compute each class's posterior $q_k(x) = p(y = k \mid x)$ and predict the largest. For two classes, equivalently, look at the **log-likelihood ratio**

$$h(x) = \ln\frac{q_1(x)}{q_0(x)} = \ln\frac{p(x \mid y=1)}{p(x\mid y=0)} + \ln\frac{p(y=1)}{p(y=0)} \quad\text{and predict 1 if } h(x) > 0$$

The boundary is where the posteriors tie. Note the second term: a **prior** that favors one class shifts the boundary toward the *other* class, so more of the space is claimed by the common class.

This rule is optimal: no classifier can have a lower error rate. Its error, the **Bayes error**, is the classification analogue of notebook 06's noise floor. It's the overlap that remains between the classes even with perfect knowledge.

🎚️ **Two 1-D Gaussian classes.** Move them apart, change their widths and the prior. The shaded area is the Bayes error: the probability mass that lands on the wrong side of the optimal boundary.

In [2]:
def plot_bayes_1d(mu1=2.0, sd0=1.0, sd1=1.0, prior1=0.5):
    xs = np.linspace(-5, 8, 1000)
    j0 = (1 - prior1) * norm.pdf(xs, 0, sd0); j1 = prior1 * norm.pdf(xs, mu1, sd1)    # joint p(x, y)
    pred1 = j1 > j0
    err = np.trapezoid(np.where(pred1, j0, j1), xs)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, gridspec_kw=dict(height_ratios=[1.6, 1]))
    ax1.plot(xs, j0, color=C["blue"], lw=2, label="p(x, y=0) = p(x|y=0) p(y=0)")
    ax1.plot(xs, j1, color=C["red"], lw=2, label="p(x, y=1) = p(x|y=1) p(y=1)")
    ax1.fill_between(xs, 0, np.where(pred1, j0, j1), color=C["ink"], alpha=0.25, label=f"Bayes error = {err:.3f}")
    for b in xs[1:][np.diff(pred1.astype(int)) != 0]:
        ax1.axvline(b, color=C["ink"], ls="--", lw=1.5); ax2.axvline(b, color=C["ink"], ls="--", lw=1.5)
    ax1.set(ylabel="density", title="Predict whichever joint density is higher"); ax1.legend(fontsize=9, loc="upper right")
    post = j1 / (j0 + j1 + 1e-300)
    ax2.plot(xs, post, color=C["violet"], lw=2.5); ax2.axhline(0.5, color=C["muted"], lw=0.8)
    ax2.set(xlabel="x", ylabel="P(y=1 | x)", ylim=(-0.05, 1.05))
    fig.tight_layout(); plt.show()

interact(plot_bayes_1d, mu1=FloatSlider(value=2.0, min=0, max=6, step=0.1, description="class-1 mean"),
         sd0=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.1, description="class-0 sd"),
         sd1=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.1, description="class-1 sd"),
         prior1=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="P(y=1)"));

---
> 🚧 **This notebook is being written section by section.** The sections above are complete and run top to bottom; more are on the way.